# ZAC 2026 LifeMate: V7 Final Inference & Full Public Test Submission (Colab)

Chạy suy luận **V7** trên toàn bộ **634 mẫu `public_test`** dựa trên code mới nhất từ `https://github.com/sun0810pi/zalo-challenge.git` (`main`, commit `b8c8f011`+):
- **Tận dụng 100% ASR Cache V6** (`/content/drive/MyDrive/zalo-challenge-v6/asr_cache/asr_public_test_whisper-turbo-vi-ct2-newly-trained.json`) -> bỏ qua ~40 phút Whisper.
- **Tự động nạp Best V7 LoRA** từ `/content/drive/MyDrive/zalo-challenge-v7/final_model_v7_final_l4_max` (hoặc `.zip` / best checkpoint).
- **Áp dụng 3 bản vá chuẩn hóa trước khi import `LifeMateAgent`**:
  1. `src/hybrid_engine.py`: Không cắt nhầm câu điều kiện thời tiết (`"nếu"`, `"trời không mưa"`, `"không mưa thì"`) ở 0 bước (sửa `public_0035`).
  2. `src/agent.py`: Tắt rule chặn nhầm `parking.reserve` sau khi đặt `ride_hailing.book(..., bike)` (sửa `public_0211`).
  3. `src/guardrail.py`: Không thu hoạch token/mã đọc miệng từ câu user (`"V21"`, `"RV157"`, `"B05"`, `"B12"`, `"SE7"`...) vào `all_known_tokens` / `user_explicit_id_tokens`, buộc gọi tool tra cứu lấy ID thật trước.
- **Checkpoint từng 10 mẫu lên Google Drive** để không bao giờ mất tiến độ nếu Colab reset RAM/GPU, xuất sẵn **`submission_v7_full_634.jsonl`** và gói audit `.zip`.
- **Không đụng vào V6**: Code clone về `/content/zalo-challenge` (SSD của VM Colab), kết quả xuất vào `/content/drive/MyDrive/zalo-challenge-v7/submissions/`.

## 1. Settings

In [ ]:
# ============================ SETTINGS ============================
SPLIT               = "public_test"     # "public_test" (634 mẫu) | "train"
N_SMOKE             = 9999              # chạy toàn bộ split (634/634 mẫu)
SELECTION           = "first"           # "first" | "stratified" | "ids"
SMOKE_IDS           = []                # dùng khi SELECTION == "ids"

GITHUB_REPO         = "https://github.com/sun0810pi/zalo-challenge.git"
CODE_DIR            = "/content/zalo-challenge"

V7_PROJECT_DIR      = "/content/drive/MyDrive/zalo-challenge-v7"
V6_PROJECT_DIR      = "/content/drive/MyDrive/zalo-challenge-v6"
DRIVE_DATA_ROOT     = ""                # "" = tự dò thư mục chứa public_test/ và models/whisper-turbo-...

LLM_BASE            = "Qwen/Qwen3-4B"
ASR_DIR_NAME        = "models/whisper-turbo-vi-ct2-newly-trained"
PREPROCESSOR        = "safe"            # "safe" = SafeContextPreprocessor | "a" = ContextLanguagePreprocessorA

BACKEND             = "vllm"            # "vllm" | "auto"
MAX_MODEL_LEN       = 16384             # giới hạn context theo thể lệ
MAX_NEW_TOKENS      = 512
MAX_STEPS           = 8
GPU_MEM_UTIL        = 0.85

FORCE_RETRANSCRIBE  = False             # giữ False để dùng ngay ASR Cache V6 (0 giây)
CHECKPOINT_EVERY    = 10                # lưu checkpoint lên Drive mỗi 10 mẫu
DETERMINISM_CHECK_N = 3                 # chạy lại kiểm tra reproducibility (0 = bỏ qua)
ALLOW_CPU           = False
RUN_TAG             = "v7"
# ==================================================================
print("V7 inference settings loaded.")

## 2. Install dependencies

In [ ]:
import subprocess, sys

PKGS = [
    "faster-whisper>=1.1.0",
    "transformers>=4.51.0",
    "peft>=0.14.0",
    "accelerate>=1.0.0",
    "scikit-learn",
    "soundfile>=0.12.1",
    "scipy",
    "pandas",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=False)
print("✅ Install step finished.")

## 3. Mount Drive, Clone/Pull GitHub `main` (`b8c8f011`+) & Link Dataset `public_test`
Tạo workspace sạch tại `/content/zalo-challenge`, giữ nguyên 100% thư mục V6 trên Google Drive.

In [ ]:
import os, sys, json, time, copy, csv, hashlib, shutil, traceback, zipfile, subprocess
from datetime import datetime
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

os.makedirs(V7_PROJECT_DIR, exist_ok=True)
CODE_ROOT = Path(CODE_DIR)

# 1. Clone hoặc cập nhật code mới nhất từ GitHub main (giữ Drive sạch)
if (CODE_ROOT / ".git").is_dir():
    print(f"Đang cập nhật {CODE_ROOT} từ GitHub main...")
    subprocess.run(["git", "-C", str(CODE_ROOT), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(CODE_ROOT), "checkout", "--", "."], check=True)
    subprocess.run(["git", "-C", str(CODE_ROOT), "reset", "--hard", "origin/main"], check=True)
else:
    if CODE_ROOT.exists():
        shutil.rmtree(CODE_ROOT)
    print(f"Đang clone {GITHUB_REPO} -> {CODE_ROOT}...")
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, str(CODE_ROOT)], check=True)

GIT_HEAD = subprocess.check_output(["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True).strip()
print(f"GitHub HEAD: {GIT_HEAD}")

# 2. Tự dò public_test/ & whisper-turbo-vi-ct2-newly-trained trên Drive
def find_drive_split_and_asr():
    cands = []
    if DRIVE_DATA_ROOT:
        cands.append(Path(DRIVE_DATA_ROOT))
    cands += [
        Path("/content/drive/MyDrive/zalo-challenge"),
        Path(V6_PROJECT_DIR),
        Path(V7_PROJECT_DIR),
        Path.cwd(),
    ]
    if Path("/content/drive/MyDrive").is_dir():
        cands += sorted(Path("/content/drive/MyDrive").glob("*"))
        cands += sorted(Path("/content/drive/MyDrive").glob("*/*"))

    found_split_dir = None
    found_asr_dir = None
    for c in cands:
        try:
            sp = c / SPLIT
            if found_split_dir is None and (sp / f"{SPLIT}.jsonl").is_file() and (sp / "zaic_tools").is_dir() and (sp / f"{SPLIT}_environment").is_dir():
                found_split_dir = sp.resolve()
            asr_cand = c / ASR_DIR_NAME
            if found_asr_dir is None and (asr_cand / "model.bin").is_file():
                found_asr_dir = asr_cand.resolve()
            asr_cand2 = c / Path(ASR_DIR_NAME).name
            if found_asr_dir is None and (asr_cand2 / "model.bin").is_file():
                found_asr_dir = asr_cand2.resolve()
        except (OSError, PermissionError):
            continue
    return found_split_dir, found_asr_dir

DRIVE_SPLIT_DIR, DRIVE_ASR_DIR = find_drive_split_and_asr()
if DRIVE_SPLIT_DIR is None:
    raise SystemExit(
        f"Không tìm thấy thư mục '{SPLIT}/' chứa {SPLIT}.jsonl, zaic_tools/, {SPLIT}_environment/ trên Drive!\n"
        f"Hãy gán DRIVE_DATA_ROOT ở ô Settings (ví dụ '/content/drive/MyDrive/zalo-challenge')."
    )

# Tạo symlink vào /content/zalo-challenge mà không copy hàng GB qua Drive
link_split = CODE_ROOT / SPLIT
if link_split.is_symlink() or link_split.is_file():
    link_split.unlink()
elif link_split.is_dir():
    shutil.rmtree(link_split)
link_split.symlink_to(DRIVE_SPLIT_DIR)

if DRIVE_ASR_DIR is not None:
    link_asr = CODE_ROOT / ASR_DIR_NAME
    link_asr.parent.mkdir(parents=True, exist_ok=True)
    if link_asr.is_symlink() or link_asr.is_file():
        link_asr.unlink()
    elif link_asr.is_dir():
        shutil.rmtree(link_asr)
    link_asr.symlink_to(DRIVE_ASR_DIR)

ROOT = CODE_ROOT
SPLIT_DIR = ROOT / SPLIT
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
HAS_CUDA = torch.cuda.is_available()
if not HAS_CUDA and not ALLOW_CPU:
    raise SystemExit("No CUDA GPU! Hãy chọn Runtime -> Change runtime type -> GPU (L4/T4).")

GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else "cpu"
print(f"Code root      : {ROOT}")
print(f"Linked split   : {SPLIT_DIR} -> {DRIVE_SPLIT_DIR}")
print(f"Linked ASR dir : {DRIVE_ASR_DIR}")
if HAS_CUDA:
    print(f"GPU            : {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB), bf16={torch.cuda.is_bf16_supported()}")

## 4. Áp Dụng 3 Bản Vá Surgical Lên `/content/zalo-challenge/src/` (Trước Khi Import `LifeMateAgent`)
1. **`src/hybrid_engine.py`**: Không hủy câu có mệnh đề điều kiện (`"nếu"`, `"trời không mưa"`, `"không mưa thì"`).
2. **`src/agent.py`**: Bỏ rule chặn nhầm `parking.reserve` sau khi đặt `ride_hailing.book(..., bike)`.
3. **`src/guardrail.py`**: Không đưa token thô/mã đọc miệng (`"V21"`, `"RV157"`, `"B05"`, `"B12"`, `"SE7"`) từ câu user vào `all_known_tokens` / `user_explicit_id_tokens`.

In [ ]:
# Đảm bảo bắt đầu từ code sạch trên repo rồi mới vá để chạy lại cell nhiều lần không bị vá trùng
subprocess.run(["git", "-C", str(ROOT), "checkout", "--", "src/hybrid_engine.py", "src/agent.py", "src/guardrail.py"], check=True)

# --- Fix (a): src/hybrid_engine.py ---
he_path = ROOT / "src" / "hybrid_engine.py"
he_txt = he_path.read_text(encoding="utf-8")
old_he = 'if any(kw in t.lower() for kw in ("chỉ cần xem", "chỉ xem", "xem giúp", "kiểm tra giúp", "coi giúp", "trừ ", "ngoại trừ ")):'
new_he = 'if any(kw in t.lower() for kw in ("chỉ cần xem", "chỉ xem", "xem giúp", "kiểm tra giúp", "coi giúp", "trừ ", "ngoại trừ ", "nếu", "trời không mưa", "không mưa thì")):'
assert old_he in he_txt, "Không tìm thấy pattern target trong src/hybrid_engine.py!"
he_path.write_text(he_txt.replace(old_he, new_he, 1), encoding="utf-8")

# --- Fix (b): src/agent.py ---
ag_path = ROOT / "src" / "agent.py"
ag_txt = ag_path.read_text(encoding="utf-8")
old_ag = 'if t_name == "parking.reserve" and any("ride_hailing.book" in s and "bike" in s for s in state.seen_calls):'
new_ag = 'if False and t_name == "parking.reserve" and any("ride_hailing.book" in s and "bike" in s for s in state.seen_calls):  # disabled V7 false-positive'
assert old_ag in ag_txt, "Không tìm thấy pattern parking.reserve trong src/agent.py!"
ag_path.write_text(ag_txt.replace(old_ag, new_ag, 1), encoding="utf-8")

# --- Fix (c): src/guardrail.py ---
gr_path = ROOT / "src" / "guardrail.py"
gr_txt = gr_path.read_text(encoding="utf-8")
old_gr = '''    def _harvest_user_id_tokens(self, text: str) -> None:
        for tok in re.findall(r"[A-Za-z0-9_]+", text):
            self.all_known_tokens.add(tok)
            if "_" in tok or any(ch.isdigit() for ch in tok):
                self.user_explicit_id_tokens.add(tok)'''
new_gr = '''    def _harvest_user_id_tokens(self, text: str) -> None:
        # V7 surgical fix: do not populate all_known_tokens or user_explicit_id_tokens from raw user text
        # so spoken codes (V21, RV157, B05, B12, SE7...) must be verified via read tools first.
        return'''
assert old_gr in gr_txt, "Không tìm thấy _harvest_user_id_tokens trong src/guardrail.py!"
gr_path.write_text(gr_txt.replace(old_gr, new_gr, 1), encoding="utf-8")

print("✅ Đã áp dụng thành công 3 bản vá trên src/!")

def patch_peft_torchao():
    done = []
    try:
        import peft.import_utils
        peft.import_utils.is_torchao_available = lambda: False
        done.append("import_utils")
    except Exception:
        pass
    try:
        import peft.tuners.lora.torchao
        peft.tuners.lora.torchao.is_torchao_available = lambda: False
        done.append("lora.torchao")
    except Exception:
        pass
    return done

print(f"peft torchao check disabled: {patch_peft_torchao() or 'ok'}")

if str(SPLIT_DIR) not in sys.path:
    sys.path.insert(0, str(SPLIT_DIR))
from zaic_tools import ToolEnv                                  # noqa: E402
from zaic_tools.validate import catalog_tools, schema_validate  # noqa: E402

from src.agent import (                                         # noqa: E402
    LifeMateAgent,
    LLMBackend,
    _resolve_max_lora_rank,
    fix_all_seeds,
    render_chatml_preserving_think,
)
from src.asr import Transcriber                                 # noqa: E402
from src.guardrail import PureSchemaStateTracker                # noqa: E402
from src.hybrid_engine import HybridProgramAidedEngine          # noqa: E402
from src.preprocessor_a import ContextLanguagePreprocessorA     # noqa: E402
from src.router import SemanticToolRouter                       # noqa: E402
from evaluate import evaluate_predictions                       # noqa: E402

# Tự kiểm tra nhanh 3 bản vá sau khi import
assert not HybridProgramAidedEngine.is_complete_cancellation(
    "Mai nếu có mưa thì nhắc anh mang áo mưa, còn trời không mưa rồi thì cứ để nguyên."
), "Fix (a) failed!"
assert HybridProgramAidedEngine.is_complete_cancellation("Thôi đừng đặt nữa, à mà thôi.")
_test_tracker = PureSchemaStateTracker({}, "Kiểm tra đơn V21 tàu SE7 chuyến RV157 phòng B05 B12", {})
assert len(_test_tracker.all_known_tokens) == 0 and len(_test_tracker.user_explicit_id_tokens) == 0, "Fix (c) failed!"
print("✅ Self-check 3 bản vá đạt 100%!")

fix_all_seeds(42)

CATALOG = json.loads((SPLIT_DIR / f"{SPLIT}_tools_catalog.json").read_text(encoding="utf-8"))
TOOLS_MAP = catalog_tools(CATALOG)
APPS_MAP = {a["id"]: a for a in CATALOG.get("apps", [])}
ALL_SAMPLES = [
    json.loads(l)
    for l in (SPLIT_DIR / f"{SPLIT}.jsonl").read_text(encoding="utf-8").splitlines()
    if l.strip()
]

def select_samples(samples):
    if SELECTION == "ids":
        by_id = {s["id"]: s for s in samples}
        missing = [i for i in SMOKE_IDS if i not in by_id]
        if missing:
            raise SystemExit(f"IDs not in {SPLIT}: {missing}")
        keep = set(SMOKE_IDS)
    elif SELECTION == "first":
        keep = {s["id"] for s in samples[:N_SMOKE]}
    elif SELECTION == "stratified":
        ordered = sorted(samples, key=lambda s: (len(s.get("history") or []), s["id"]))
        n = min(N_SMOKE, len(ordered))
        pos = {round(i * (len(ordered) - 1) / max(n - 1, 1)) for i in range(n)}
        keep = {ordered[p]["id"] for p in pos}
    else:
        raise SystemExit(f"Unknown SELECTION {SELECTION!r}")
    return [s for s in samples if s["id"] in keep]

SMOKE = select_samples(ALL_SAMPLES)
print(f"{len(SMOKE)} of {len(ALL_SAMPLES)} samples selected ({SELECTION}) | {len(APPS_MAP)} apps | {len(TOOLS_MAP)} tools")

## 5. Tìm & Giải Nén Best V7 LoRA Adapter
Tự tìm `/content/drive/MyDrive/zalo-challenge-v7/final_model_v7_final_l4_max`, hoặc `qwen3_4b_zac_lora_v7_final_l4_max.zip`, hoặc best checkpoint trong `checkpoints_v7_final_l4_max`.

In [ ]:
LORA_DIR = Path("/content/zac_lora/v7_final_l4_max")
V7_FINAL_DIR = Path(V7_PROJECT_DIR) / "final_model_v7_final_l4_max"
V7_ZIP_PATH = Path(V7_PROJECT_DIR) / "qwen3_4b_zac_lora_v7_final_l4_max.zip"
V7_CKPT_ROOT = Path(V7_PROJECT_DIR) / "checkpoints_v7_final_l4_max"

LORA_SOURCE = ""
LORA_NAME = "v7_final_l4_max"

def _is_valid_lora_dir(p: Path) -> bool:
    return p.is_dir() and (p / "adapter_config.json").is_file() and (p / "adapter_model.safetensors").is_file()

def _copy_dir_to_local(src: Path, dst: Path):
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)

if _is_valid_lora_dir(V7_FINAL_DIR):
    print(f"Tìm thấy V7 final model dir: {V7_FINAL_DIR}")
    _copy_dir_to_local(V7_FINAL_DIR, LORA_DIR)
    LORA_SOURCE = str(V7_FINAL_DIR)
elif V7_ZIP_PATH.is_file():
    print(f"Tìm thấy V7 zip: {V7_ZIP_PATH}, đang giải nén -> {LORA_DIR}...")
    if LORA_DIR.exists():
        shutil.rmtree(LORA_DIR)
    LORA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(V7_ZIP_PATH) as zf:
        infos = {i.filename: i for i in zf.infolist() if not i.is_dir()}
        cfg_name = next((n for n in infos if n.endswith("adapter_config.json")), None)
        assert cfg_name is not None, f"No adapter_config.json in {V7_ZIP_PATH}"
        prefix = cfg_name[: -len("adapter_config.json")]
        for n, info in infos.items():
            if not n.startswith(prefix):
                continue
            dest = LORA_DIR / n[len(prefix):]
            dest.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(dest, "wb") as dst:
                shutil.copyfileobj(src, dst, 16 << 20)
    LORA_SOURCE = str(V7_ZIP_PATH)
elif V7_CKPT_ROOT.is_dir():
    ckpts = sorted(
        [p for p in V7_CKPT_ROOT.glob("checkpoint-*") if _is_valid_lora_dir(p)],
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )
    if not ckpts:
        raise SystemExit(f"Không tìm thấy checkpoint LoRA hợp lệ nào trong {V7_CKPT_ROOT}!")
    best_ckpt = None
    for p in reversed(ckpts):
        ts = p / "trainer_state.json"
        if ts.is_file():
            try:
                bmc = json.loads(ts.read_text(encoding="utf-8")).get("best_model_checkpoint")
                if bmc and _is_valid_lora_dir(Path(bmc)):
                    best_ckpt = Path(bmc)
                    break
            except Exception:
                pass
    chosen = best_ckpt or ckpts[-1]
    print(f"Fallback từ checkpoints V7 -> chọn: {chosen}")
    _copy_dir_to_local(chosen, LORA_DIR)
    LORA_SOURCE = str(chosen)
else:
    raise SystemExit(
        f"Không tìm thấy V7 LoRA ở {V7_FINAL_DIR}, {V7_ZIP_PATH}, hay {V7_CKPT_ROOT}!\n"
        f"Hãy chạy xong notebook Train_Qwen3_4B_ZAIC2026_V7_L4_Max.ipynb trước."
    )

w_local = LORA_DIR / "adapter_model.safetensors"
assert _is_valid_lora_dir(LORA_DIR), f"Adapter không hợp lệ tại {LORA_DIR}"
ADAPTER_CFG = json.loads((LORA_DIR / "adapter_config.json").read_text(encoding="utf-8"))
TRAIN_SUMMARY = {}
if (LORA_DIR / "training_summary.json").is_file():
    TRAIN_SUMMARY = json.loads((LORA_DIR / "training_summary.json").read_text(encoding="utf-8"))
    TRAIN_SUMMARY.pop("log_history", None)
TRAIN_MAX_SEQ = int(TRAIN_SUMMARY.get("max_seq_len") or 9216)

bad = []
if ADAPTER_CFG.get("peft_type") != "LORA":
    bad.append(f"peft_type={ADAPTER_CFG.get('peft_type')}, expected LORA")
if ADAPTER_CFG.get("use_dora"):
    bad.append("DoRA adapter: vLLM cannot serve it")
if ADAPTER_CFG.get("base_model_name_or_path") != LLM_BASE:
    bad.append(f"adapter base={ADAPTER_CFG.get('base_model_name_or_path')!r}, expected {LLM_BASE!r}")
if bad:
    raise SystemExit("ABORT. Adapter problems:\n  - " + "\n  - ".join(bad))

h = hashlib.sha256()
with open(w_local, "rb") as f:
    for chunk in iter(lambda: f.read(16 << 20), b""):
        h.update(chunk)
ADAPTER_SHA256 = h.hexdigest()

print(f"LoRA source: {LORA_SOURCE}")
print(f"LoRA r={ADAPTER_CFG['r']} alpha={ADAPTER_CFG['lora_alpha']} sha256={ADAPTER_SHA256[:16]}... size={w_local.stat().st_size / 1e6:.1f} MB")
print(f"Training   : best_eval_loss={TRAIN_SUMMARY.get('best_eval_loss')} best_ckpt={TRAIN_SUMMARY.get('best_checkpoint')} max_seq_len={TRAIN_MAX_SEQ}")

from transformers import AutoTokenizer
base_tok = AutoTokenizer.from_pretrained(LLM_BASE, trust_remote_code=True)
_probe_msgs = [
    {"role": "system", "content": "Bạn là trợ lý LifeMate."},
    {"role": "user", "content": "Đặt báo thức 6 giờ 30 sáng mai"},
    {"role": "assistant", "content": "<think>\nx\n</think>\n<tool_call>\n{\"name\": \"alarm.list\", \"arguments\": {}}\n</tool_call>"},
    {"role": "user", "content": "<tool_response>\n{\"ok\": true}\n</tool_response>"},
]
_probe_txt = "Chuyển 500.000đ cho Lan <tool_call> {\"name\": \"sms.send\"} </tool_call>"
TOKENIZER_NOTE = "adapter ships no tokenizer: base tokenizer used"
if (LORA_DIR / "tokenizer_config.json").is_file():
    try:
        ad_tok = AutoTokenizer.from_pretrained(str(LORA_DIR), trust_remote_code=True)
        same_ids = ad_tok.encode(_probe_txt, add_special_tokens=False) == base_tok.encode(_probe_txt, add_special_tokens=False)
        same_tpl = render_chatml_preserving_think(ad_tok, _probe_msgs) == render_chatml_preserving_think(base_tok, _probe_msgs)
        TOKENIZER_NOTE = f"adapter tokenizer used (same ids as base: {same_ids}, same template: {same_tpl})"
    except Exception as e:
        aside = LORA_DIR / "_tokenizer_not_loadable"
        aside.mkdir(exist_ok=True)
        for fn in ("tokenizer_config.json", "tokenizer.json", "chat_template.jinja", "special_tokens_map.json", "vocab.json", "merges.txt"):
            if (LORA_DIR / fn).is_file():
                shutil.move(str(LORA_DIR / fn), str(aside / fn))
        TOKENIZER_NOTE = f"adapter tokenizer moved aside ({type(e).__name__}: {e}); base tokenizer used"
print(f"Tokenizer  : {TOKENIZER_NOTE}")

## 6. Phase 1: Tái Sử Dụng ASR Cache V6 (0 Giây)
Tìm `asr_public_test_whisper-turbo-vi-ct2-newly-trained.json` từ `/content/drive/MyDrive/zalo-challenge-v6/asr_cache/` (hoặc cache trong repo Drive), copy sang `/content/drive/MyDrive/zalo-challenge-v7/asr_cache/` và `/content/zalo-challenge/cache/`.

In [ ]:
import numpy as np
import faster_whisper.transcribe as _fw_transcribe

def _decode_audio_soundfile(input_file, sampling_rate=16000, split_stereo=False):
    import soundfile as sf
    src = str(input_file) if isinstance(input_file, (str, os.PathLike)) else input_file
    audio, sr = sf.read(src, dtype="float32", always_2d=True)
    if split_stereo and audio.shape[1] == 2:
        return np.ascontiguousarray(audio[:, 0]), np.ascontiguousarray(audio[:, 1])
    audio = audio.mean(axis=1)
    if sr != sampling_rate:
        from math import gcd
        from scipy.signal import resample_poly
        g = gcd(int(sr), int(sampling_rate))
        audio = resample_poly(audio, sampling_rate // g, sr // g)
    return np.ascontiguousarray(audio, dtype=np.float32)

_fw_transcribe.decode_audio = _decode_audio_soundfile

ASR_TAG = Path(ASR_DIR_NAME).name
V7_ASR_CACHE = Path(V7_PROJECT_DIR) / "asr_cache" / f"asr_{SPLIT}_{ASR_TAG}.json"
V7_ASR_CACHE.parent.mkdir(parents=True, exist_ok=True)
REPO_ASR_CACHE = ROOT / "cache" / f"asr_{SPLIT}_{ASR_TAG}.json"
REPO_ASR_CACHE.parent.mkdir(parents=True, exist_ok=True)

CANDIDATE_CACHES = [
    V7_ASR_CACHE,
    Path(V6_PROJECT_DIR) / "asr_cache" / f"asr_{SPLIT}_{ASR_TAG}.json",
    DRIVE_SPLIT_DIR.parent / "cache" / f"smoke_asr_{SPLIT}_{ASR_TAG}.json",
    DRIVE_SPLIT_DIR.parent / "cache" / f"asr_{SPLIT}_{ASR_TAG}.json",
    DRIVE_SPLIT_DIR.parent / "cache" / f"asr_{SPLIT}.json",
]

if not FORCE_RETRANSCRIBE:
    src_cache = next((p for p in CANDIDATE_CACHES if p.is_file() and p.stat().st_size > 100), None)
    if src_cache is not None:
        if src_cache != V7_ASR_CACHE:
            shutil.copy2(src_cache, V7_ASR_CACHE)
        shutil.copy2(V7_ASR_CACHE, REPO_ASR_CACHE)
        print(f"✅ Đã tái sử dụng ASR cache từ: {src_cache} -> {REPO_ASR_CACHE}")
    else:
        print("⚠️ Không tìm thấy ASR cache cũ; sẽ chạy Whisper pre-transcribe nếu thiếu mẫu.")
else:
    if REPO_ASR_CACHE.is_file():
        REPO_ASR_CACHE.unlink()

asr_model_arg = str(ROOT / ASR_DIR_NAME) if (ROOT / ASR_DIR_NAME / "model.bin").is_file() else ASR_DIR_NAME
transcriber = Transcriber(
    model_name=asr_model_arg,
    device="cuda" if HAS_CUDA else "cpu",
    compute_type="float16" if HAS_CUDA else "int8",
    cache_path=REPO_ASR_CACHE,
)
t0 = time.perf_counter()
transcriber.pretranscribe_samples(SMOKE, SPLIT_DIR)
ASR_BACKEND = transcriber._backend or "cached"
if ASR_BACKEND not in ("faster_whisper", "cached"):
    raise SystemExit(f"ABORT. ASR fell back to {ASR_BACKEND!r}, check {asr_model_arg}")

ASR = {s["id"]: transcriber.transcribe_sample(s, SPLIT_DIR) for s in SMOKE}
transcriber.save_cache()
shutil.copy2(REPO_ASR_CACHE, V7_ASR_CACHE)
transcriber.unload_model()
ASR_SECONDS = round(time.perf_counter() - t0, 1)

print(f"\n✅ ASR ready ({ASR_BACKEND}) in {ASR_SECONDS}s | samples={len(ASR)} | cache={V7_ASR_CACHE}")

## 7. Phase 2: Load `Qwen/Qwen3-4B` + V7 LoRA & Preprocessor

In [ ]:
llm = LLMBackend(
    model_name_or_path=LLM_BASE,
    lora_path=str(LORA_DIR),
    backend=BACKEND,
    max_model_len=MAX_MODEL_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    gpu_memory_utilization=GPU_MEM_UTIL,
)
if not llm.lora_path:
    raise SystemExit(f"ABORT. LoRA was not attached ({LORA_DIR}).")
VLLM_MAX_LORA_RANK = _resolve_max_lora_rank(llm.lora_path, override=llm.max_lora_rank)
print(f"LLM backend: {llm.backend} | LoRA: {llm.lora_path} | max_lora_rank={VLLM_MAX_LORA_RANK}")

if PREPROCESSOR == "safe":
    from src.asr_fallback import SafeContextPreprocessor
    _safe = SafeContextPreprocessor(sim_threshold=0.80)

    class _SafePreprocessorAdapter:
        def preprocess(self, raw_asr_text, sample):
            return _safe.preprocess_with_evidence(raw_asr_text, sample)[0]

    preprocessor = _SafePreprocessorAdapter()
elif PREPROCESSOR == "a":
    preprocessor = ContextLanguagePreprocessorA()
else:
    raise SystemExit(f"Unknown PREPROCESSOR {PREPROCESSOR!r}")
print(f"Preprocessor: {PREPROCESSOR}")

## 8. Chạy Agent Trên 634 Mẫu & Checkpoint Từng 10 Mẫu Lên Google Drive
Nếu Colab bị ngắt kết nối hoặc reset RAM giữa chừng, chỉ cần **Run all** lại — cell này tự đọc `CKPT_JSONL` trên Drive và chỉ chạy tiếp những mẫu còn thiếu!

In [ ]:
PROMPT_BUDGET = MAX_MODEL_LEN - MAX_NEW_TOKENS - 32

class RecordingLLM:
    def __init__(self, inner):
        self.inner = inner
        self.steps = []

    def generate(self, messages, temperature=0.0):
        prompt = render_chatml_preserving_think(self.inner.tokenizer, messages, add_generation_prompt=True)
        n_tok = len(self.inner.tokenizer.encode(prompt, add_special_tokens=False))
        t0 = time.perf_counter()
        reply = self.inner.generate(messages, temperature=temperature)
        self.steps.append({
            "prompt_tokens": n_tok,
            "truncated": n_tok > PROMPT_BUDGET,
            "seconds": round(time.perf_counter() - t0, 2),
            "reply": reply,
        })
        return reply

class AuditEnv:
    def __init__(self, env):
        self._env = env
        self.executions = []
        self.validation_errors = []

    def call(self, name, arguments=None):
        res = self._env.call(name, arguments)
        self.executions.append({"name": name, "arguments": copy.deepcopy(arguments), "result": res})
        return res

    def validate(self, name, arguments=None):
        v = self._env.validate(name, arguments)
        if not v.get("ok"):
            self.validation_errors.append({
                "name": name,
                "arguments": copy.deepcopy(arguments),
                "error": v.get("error"),
            })
        return v

    def __getattr__(self, attr):
        return getattr(self._env, attr)

REC = RecordingLLM(llm)
AGENT = LifeMateAgent(
    llm=REC,
    catalog_tools_map=TOOLS_MAP,
    max_steps=MAX_STEPS,
    preprocessor=preprocessor,
    router=SemanticToolRouter(top_k_recommended=4),
    catalog_apps_map=APPS_MAP,
)

def run_one(item):
    sid = item["id"]
    final_text, hist = ASR[sid]
    initial_hash = ToolEnv(SPLIT, sid, root=ROOT).get_state_hash()
    env = AuditEnv(ToolEnv(SPLIT, sid, root=ROOT))
    REC.steps = []
    err = None
    t0 = time.perf_counter()
    try:
        tool_calls, state_hash = AGENT.run_sample(
            env=env,
            sample=item,
            final_user_text=final_text,
            history_transcripts=hist,
        )
    except Exception:
        err = traceback.format_exc()
        tool_calls, state_hash = env.get_tool_calls(), env.get_state_hash()
    seconds = round(time.perf_counter() - t0, 2)

    cleaned = preprocessor.preprocess(final_text, item)
    schema_errs = [v for v in schema_validate(tool_calls, CATALOG, env.allowed) if not v["ok"]]
    rejected = [e for e in env.executions if isinstance(e["result"], dict) and not e["result"].get("ok")]
    noops = [e for e in env.executions if isinstance(e["result"], dict) and e["result"].get("ok") and e["result"].get("done") is False]
    writes = [e for e in env.executions if (TOOLS_MAP.get(e["name"]) or {}).get("kind") == "write"]
    steps = list(REC.steps)
    max_tok = max((s["prompt_tokens"] for s in steps), default=0)

    flags = []
    if err:
        flags.append("EXCEPTION")
    if not (cleaned or "").strip():
        flags.append("EMPTY_TRANSCRIPT")
    if not tool_calls:
        flags.append("NO_TOOL_CALLS")
    if schema_errs:
        flags.append(f"SCHEMA_ERROR:{len(schema_errs)}")
    if rejected:
        flags.append(f"REJECTED_CALLS:{len(rejected)}")
    if noops:
        flags.append(f"SOFT_NOOP:{len(noops)}")
    if writes and state_hash == initial_hash:
        flags.append("WRITE_BUT_STATE_UNCHANGED")
    if env.validation_errors:
        flags.append(f"VALIDATION_RETRIES:{len(env.validation_errors)}")
    if len(steps) >= MAX_STEPS:
        flags.append("HIT_MAX_STEPS")
    if any(s["truncated"] for s in steps):
        flags.append("PROMPT_TRUNCATED")
    elif max_tok > TRAIN_MAX_SEQ:
        flags.append("PROMPT_OVER_TRAIN_LEN")

    return {
        "id": sid,
        "audio": item.get("audio"),
        "history_turns": len(item.get("history") or []),
        "installed_apps": [a for a in ((item.get("context") or {}).get("installed_apps") or {}) if a != "sys"],
        "current_time": (item.get("context") or {}).get("current_time"),
        "asr_text": final_text,
        "preprocessed_text": cleaned,
        "history_asr": hist,
        "tool_calls": tool_calls,
        "state_hash": state_hash,
        "initial_state_hash": initial_hash,
        "state_changed": state_hash != initial_hash,
        "executions": env.executions,
        "validation_errors": env.validation_errors,
        "schema_errors": schema_errs,
        "llm_steps": steps,
        "max_prompt_tokens": max_tok,
        "seconds": seconds,
        "error": err,
        "flags": flags,
    }

CKPT_DIR = Path(V7_PROJECT_DIR) / "infer_checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_JSONL = CKPT_DIR / f"records_{SPLIT}_{RUN_TAG}_{ADAPTER_SHA256[:8]}.jsonl"

done_map = {}
if CKPT_JSONL.is_file():
    for line in CKPT_JSONL.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
            if rec.get("id"):
                done_map[rec["id"]] = rec
        except json.JSONDecodeError:
            continue

def _flush_checkpoint_buffer(buf):
    if not buf:
        return
    with open(CKPT_JSONL, "a", encoding="utf-8") as f:
        for rec in buf:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
    buf.clear()

missing = [s for s in SMOKE if s["id"] not in done_map]
print(f"Checkpoint Drive ({CKPT_JSONL.name}): đã xong {len(done_map)}/{len(SMOKE)} mẫu | Còn thiếu: {len(missing)} mẫu")

t_run = time.perf_counter()
ckpt_buf = []
try:
    for idx, item in enumerate(missing, len(done_map) + 1):
        r = run_one(item)
        done_map[r["id"]] = r
        ckpt_buf.append(r)
        if len(ckpt_buf) >= CHECKPOINT_EVERY:
            _flush_checkpoint_buffer(ckpt_buf)
        names = ", ".join(c["name"] for c in r["tool_calls"]) or "-"
        print(f"[{idx:>3}/{len(SMOKE)}] {r['id']} {r['seconds']:>5.1f}s  {len(r['llm_steps'])} steps  {len(r['tool_calls'])} calls: {names}")
        if r["flags"]:
            print(f"           FLAGS: {', '.join(r['flags'])}")
finally:
    _flush_checkpoint_buffer(ckpt_buf)

RUN_SECONDS = round(time.perf_counter() - t_run, 1)
RECORDS = [done_map[s["id"]] for s in SMOKE]
assert len(RECORDS) == len(SMOKE)
print(f"\n✅ Hoàn tất đủ {len(RECORDS)}/{len(SMOKE)} mẫu (phiên chạy mới {len(missing)} mẫu trong {RUN_SECONDS}s)!")

## 9. Determinism check

In [ ]:
DETERMINISM = []
if DETERMINISM_CHECK_N > 0:
    by_id = {s["id"]: s for s in SMOKE}
    pick = [r for r in RECORDS if r["tool_calls"]][:DETERMINISM_CHECK_N] or RECORDS[:DETERMINISM_CHECK_N]
    for r in pick:
        again = run_one(by_id[r["id"]])
        same = (again["tool_calls"] == r["tool_calls"]) and (again["state_hash"] == r["state_hash"])
        DETERMINISM.append({"id": r["id"], "identical": same})
        if not same:
            r["flags"].append("NONDETERMINISTIC")
            r["rerun_tool_calls"] = again["tool_calls"]
        print(f"  {r['id']}: {'identical' if same else 'DIFFERENT'}")
    print(f"Determinism: {sum(d['identical'] for d in DETERMINISM)}/{len(DETERMINISM)} identical")
else:
    print("Skipped (DETERMINISM_CHECK_N = 0).")

## 10. Write `submission_v7_full_634.jsonl`, Audit Pack & Replay-Check

In [ ]:
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path(V7_PROJECT_DIR) / "submissions" / f"v7_{STAMP}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
READY_SUB_PATH = Path(V7_PROJECT_DIR) / "submissions" / "submission_v7_full_634.jsonl"

REPLAY_MISMATCHES = []
for r in RECORDS:
    env_r = ToolEnv(SPLIT, r["id"], root=ROOT)
    for c in r["tool_calls"]:
        env_r.call(c["name"], c.get("arguments") or {})
    if env_r.get_state_hash() != r["state_hash"]:
        REPLAY_MISMATCHES.append(r["id"])
        r["flags"].append("REPLAY_HASH_MISMATCH")
print(f"Replay hash mismatches: {len(REPLAY_MISMATCHES)} {REPLAY_MISMATCHES or ''}")

sub_p = OUT_DIR / "submission_v7_full_634.jsonl"
with open(sub_p, "w", encoding="utf-8") as f:
    for r in RECORDS:
        f.write(json.dumps({"id": r["id"], "tool_calls": r["tool_calls"], "state_hash": r["state_hash"]}, ensure_ascii=False) + "\n")
shutil.copy2(sub_p, READY_SUB_PATH)

with open(OUT_DIR / "audit_trace.jsonl", "w", encoding="utf-8") as f:
    for r in RECORDS:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(OUT_DIR / "summary.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "history_turns", "n_calls", "tools", "state_changed", "llm_steps", "max_prompt_tokens", "seconds", "flags", "asr_text", "preprocessed_text"])
    for r in RECORDS:
        w.writerow([
            r["id"], r["history_turns"], len(r["tool_calls"]),
            " | ".join(c["name"] for c in r["tool_calls"]), r["state_changed"],
            len(r["llm_steps"]), r["max_prompt_tokens"], r["seconds"], " ".join(r["flags"]),
            r["asr_text"], r["preprocessed_text"],
        ])

print("Replay check with evaluate.py:")
EVAL = evaluate_predictions(str(sub_p), root=ROOT, split=SPLIT) or {}

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

MANIFEST = {
    "created": STAMP,
    "git_head": GIT_HEAD,
    "split": SPLIT,
    "selection": SELECTION,
    "sample_count": len(RECORDS),
    "llm": {
        "base": LLM_BASE, "backend": llm.backend, "max_lora_rank": VLLM_MAX_LORA_RANK,
        "max_model_len": MAX_MODEL_LEN, "max_new_tokens": MAX_NEW_TOKENS, "max_steps": MAX_STEPS,
        "gpu_mem_util": GPU_MEM_UTIL,
    },
    "adapter": {
        "source": LORA_SOURCE, "local_dir": str(LORA_DIR), "sha256": ADAPTER_SHA256,
        "r": ADAPTER_CFG.get("r"), "lora_alpha": ADAPTER_CFG.get("lora_alpha"),
        "target_modules": sorted(ADAPTER_CFG.get("target_modules") or []),
        "training_summary": TRAIN_SUMMARY, "tokenizer": TOKENIZER_NOTE,
    },
    "asr": {"model_dir": ASR_DIR_NAME, "backend": ASR_BACKEND, "cache": str(V7_ASR_CACHE), "seconds": ASR_SECONDS},
    "preprocessor": PREPROCESSOR,
    "v7_surgical_patches": ["hybrid_engine.is_complete_cancellation_if", "agent.parking_reserve_reenabled", "guardrail.no_raw_user_token_harvest"],
    "hardware": {"gpu": GPU_NAME, "colab": IN_COLAB},
    "packages": {m: _ver(m) for m in ("torch", "vllm", "transformers", "peft", "faster_whisper", "ctranslate2")},
    "timing": {"asr_seconds": ASR_SECONDS, "agent_seconds": RUN_SECONDS},
    "determinism": DETERMINISM,
    "replay_hash_mismatches": REPLAY_MISMATCHES,
    "evaluate": EVAL,
}
(OUT_DIR / "run_manifest.json").write_text(json.dumps(MANIFEST, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n✅ File nộp Public Submissions: {READY_SUB_PATH}")
print(f"✅ Audit pack lưu tại          : {OUT_DIR}")

## 11. Audit report & Tổng Kết

In [ ]:
def _j(x):
    return json.dumps(x, ensure_ascii=False)

n = len(RECORDS)
n_calls = sum(len(r["tool_calls"]) for r in RECORDS)
flag_counts = {}
for r in RECORDS:
    for fl in r["flags"]:
        key = fl.split(":")[0]
        flag_counts[key] = flag_counts.get(key, 0) + 1
n_schema = sum(len(r["schema_errors"]) for r in RECORDS)

lines = [
    f"# V7 Inference Report ({RUN_TAG}): {STAMP}",
    "",
    f"- GitHub HEAD: `{GIT_HEAD}`",
    f"- Split: `{SPLIT}`, {n} samples",
    f"- LLM: `{LLM_BASE}` + `{LORA_NAME}` (r={ADAPTER_CFG.get('r')}, sha256 `{ADAPTER_SHA256[:16]}`), backend `{llm.backend}`",
    f"- ASR cache: `{V7_ASR_CACHE}` ({ASR_BACKEND}), preprocessor `{PREPROCESSOR}`",
    f"- GPU: {GPU_NAME}; ASR {ASR_SECONDS}s, agent {RUN_SECONDS}s",
    "",
    "## Totals",
    "",
    "| Metric | Value |",
    "|---|---|",
    f"| Tool calls | {n_calls} (avg {n_calls / max(n, 1):.2f}/sample) |",
    f"| Samples with 0 calls | {sum(1 for r in RECORDS if not r['tool_calls'])} |",
    f"| Samples whose state changed | {sum(r['state_changed'] for r in RECORDS)} |",
    f"| Schema errors (must be 0) | {n_schema} |",
    f"| Replay hash mismatches (must be 0) | {len(REPLAY_MISMATCHES)} |",
    f"| Determinism | {sum(d['identical'] for d in DETERMINISM)}/{len(DETERMINISM)} identical |",
    f"| Max prompt tokens | {max((r['max_prompt_tokens'] for r in RECORDS), default=0)} (budget {PROMPT_BUDGET}) |",
]
if SPLIT == "train" and EVAL:
    lines += [
        f"| State accuracy | {EVAL.get('state_acc', 0) * 100:.1f}% |",
        f"| Tool-call F1 | {EVAL.get('tool_call_f1', 0) * 100:.1f}% |",
        f"| Combined | {EVAL.get('combined_score', 0) * 100:.1f}% |",
    ]
lines += ["", "## Flags", ""]
lines += [f"- `{k}`: {v} sample(s)" for k, v in sorted(flag_counts.items())] or ["- none"]
(OUT_DIR / "audit_report.md").write_text("\n".join(lines), encoding="utf-8")

print("=" * 64)
print(f"Samples completed  : {n}/{len(SMOKE)}")
print(f"Schema errors      : {n_schema}   <- must be 0")
print(f"Hash mismatches    : {len(REPLAY_MISMATCHES)}   <- must be 0")
print(f"Determinism        : {sum(d['identical'] for d in DETERMINISM)}/{len(DETERMINISM)} identical")
print(f"Flag counts        : {flag_counts or 'none'}")
print(f"Submission file    : {READY_SUB_PATH}")
print(f"Audit pack         : {OUT_DIR}")
print("=" * 64)

## 12. Zip Audit Pack & Tải File Submission Về Máy

In [ ]:
zip_out = shutil.make_archive(str(OUT_DIR), "zip", root_dir=OUT_DIR)
print(f"✅ Zipped audit pack: {zip_out}")
try:
    from google.colab import files
    files.download(str(READY_SUB_PATH))
    files.download(zip_out)
except Exception as e:
    print(f"Không tự tải xuống từ trình duyệt ({e}); hãy lấy trực tiếp trên Google Drive: {READY_SUB_PATH}")